# Advanced Problems with Solutions: What's New in Python 3.6

This notebook contains advanced practice problems based on selected Python 3.6 features:

- Dictionary insertion order
- Preserved order of `**kwargs`
- Numeric literal underscores
- f-strings
- Type annotations
- Async language enhancements

Each problem includes a complete solution and a small verification section.

## Problem 1: Ordered Configuration Merger

### Task

Write a function `merge_configs(*configs)` that merges multiple dictionaries into one dictionary while preserving the insertion order of first appearance.

Rules:

1. If a key appears for the first time, keep its position.
2. If the same key appears later, update its value but do not move its position.
3. The returned object should be a regular `dict`.

### Example

```python
merge_configs(
    {'host': 'localhost', 'port': 8000},
    {'debug': True, 'port': 9000},
    {'workers': 4, 'host': 'prod.example.com'}
)
```

Expected order:

```python
['host', 'port', 'debug', 'workers']
```

In [1]:
def merge_configs(*configs):
    """Merge dictionaries while preserving first-insertion key order.

    Later dictionaries override values, but existing keys are not moved.
    """
    merged = {}
    for config in configs:
        for key, value in config.items():
            merged[key] = value
    return merged


# Verification
result = merge_configs(
    {'host': 'localhost', 'port': 8000},
    {'debug': True, 'port': 9000},
    {'workers': 4, 'host': 'prod.example.com'}
)

assert list(result.keys()) == ['host', 'port', 'debug', 'workers']
assert result == {
    'host': 'prod.example.com',
    'port': 9000,
    'debug': True,
    'workers': 4
}

result

{'host': 'prod.example.com', 'port': 9000, 'debug': True, 'workers': 4}

## Problem 2: Ordered `**kwargs` Schema Builder

### Task

Write a function `schema(**fields)` that returns a list of dictionaries describing fields in the exact order in which they were passed.

Each output item should contain:

- `name`: the field name
- `type`: the type object passed as the value
- `required`: `True` by default

Then write another function `optional_schema(**fields)` where the passed value may be either:

- a type, meaning required
- a tuple `(type, required_bool)`

### Example

```python
optional_schema(id=int, name=str, email=(str, False))
```

In [2]:
def schema(**fields):
    return [
        {'name': name, 'type': field_type, 'required': True}
        for name, field_type in fields.items()
    ]


def optional_schema(**fields):
    result = []

    for name, spec in fields.items():
        if isinstance(spec, tuple):
            field_type, required = spec
        else:
            field_type, required = spec, True

        result.append({
            'name': name,
            'type': field_type,
            'required': required
        })

    return result


# Verification
basic = schema(id=int, name=str, active=bool)
advanced = optional_schema(id=int, name=str, email=(str, False))

assert [item['name'] for item in basic] == ['id', 'name', 'active']
assert [item['name'] for item in advanced] == ['id', 'name', 'email']
assert advanced[-1]['required'] is False

advanced

[{'name': 'id', 'type': int, 'required': True},
 {'name': 'name', 'type': str, 'required': True},
 {'name': 'email', 'type': str, 'required': False}]

## Problem 3: Human-Readable Binary Permission Parser

### Task

Python 3.6 allows underscores in numeric literals. Use this feature to make bit masks more readable.

Create constants for the following permissions:

- `READ = 0b_0001`
- `WRITE = 0b_0010`
- `EXECUTE = 0b_0100`
- `ADMIN = 0b_1000`

Write:

1. `has_permission(mask, permission)`
2. `grant_permission(mask, permission)`
3. `revoke_permission(mask, permission)`
4. `describe_permissions(mask)`

`describe_permissions` should return permission names in this order:

```python
['READ', 'WRITE', 'EXECUTE', 'ADMIN']
```

In [3]:
READ = 0b_0001
WRITE = 0b_0010
EXECUTE = 0b_0100
ADMIN = 0b_1000

PERMISSIONS = {
    'READ': READ,
    'WRITE': WRITE,
    'EXECUTE': EXECUTE,
    'ADMIN': ADMIN
}


def has_permission(mask, permission):
    return (mask & permission) == permission


def grant_permission(mask, permission):
    return mask | permission


def revoke_permission(mask, permission):
    return mask & ~permission


def describe_permissions(mask):
    return [
        name
        for name, permission in PERMISSIONS.items()
        if has_permission(mask, permission)
    ]


# Verification
mask = 0b_0000
mask = grant_permission(mask, READ)
mask = grant_permission(mask, ADMIN)
mask = grant_permission(mask, WRITE)
mask = revoke_permission(mask, ADMIN)

assert has_permission(mask, READ)
assert has_permission(mask, WRITE)
assert not has_permission(mask, ADMIN)
assert describe_permissions(mask) == ['READ', 'WRITE']

describe_permissions(mask)

['READ', 'WRITE']

## Problem 4: f-string Diagnostic Formatter

### Task

Write a function `format_diagnostics(records)` that receives a list of dictionaries with this structure:

```python
{
    'name': 'task-a',
    'duration': 1.23456,
    'memory': 1536000,
    'success': True
}
```

Return a multiline string where each line is formatted as:

```text
task-a     | duration=1.235s | memory=1,536,000 bytes | status=OK
```

Requirements:

1. Use f-strings.
2. Left-align the task name in a 10-character column.
3. Format duration to exactly 3 decimal places.
4. Format memory with comma separators.
5. Convert success to `OK` or `FAIL`.

In [4]:
def format_diagnostics(records):
    lines = []

    for record in records:
        name = record['name']
        duration = record['duration']
        memory = record['memory']
        status = 'OK' if record['success'] else 'FAIL'

        lines.append(
            f"{name:<10} | duration={duration:.3f}s | "
            f"memory={memory:,} bytes | status={status}"
        )

    return '\n'.join(lines)


# Verification
records = [
    {'name': 'task-a', 'duration': 1.23456, 'memory': 1536000, 'success': True},
    {'name': 'worker-42', 'duration': 0.1, 'memory': 4096, 'success': False}
]

output = format_diagnostics(records)

assert 'task-a     | duration=1.235s | memory=1,536,000 bytes | status=OK' in output
assert 'worker-42  | duration=0.100s | memory=4,096 bytes | status=FAIL' in output

print(output)

task-a     | duration=1.235s | memory=1,536,000 bytes | status=OK
worker-42  | duration=0.100s | memory=4,096 bytes | status=FAIL


## Problem 5: Type-Annotated Data Pipeline

### Task

Write a type-annotated function `normalize_scores(scores)`.

Input:

```python
scores: List[float]
```

Output:

```python
List[float]
```

Rules:

1. Normalize scores into the range `[0.0, 1.0]`.
2. If the input is empty, return an empty list.
3. If all scores are the same, return a list of `1.0` values.
4. Include type annotations for all helper functions.

Then inspect the annotations using `.__annotations__`.

In [5]:
from typing import List


def score_range(scores: List[float]) -> float:
    return max(scores) - min(scores)


def normalize_scores(scores: List[float]) -> List[float]:
    if not scores:
        return []

    minimum = min(scores)
    spread = score_range(scores)

    if spread == 0:
        return [1.0 for _ in scores]

    return [(score - minimum) / spread for score in scores]


# Verification
assert normalize_scores([]) == []
assert normalize_scores([5.0, 5.0]) == [1.0, 1.0]
assert normalize_scores([10.0, 20.0, 30.0]) == [0.0, 0.5, 1.0]

normalize_scores.__annotations__

{'scores': typing.List[float], 'return': typing.List[float]}

## Problem 6: Runtime Type Validation Using Annotations

### Task

Python itself does not enforce type annotations at runtime.

Write a decorator `enforce_simple_types(func)` that checks simple annotations at runtime for positional and keyword arguments.

Scope:

- Only enforce annotations that are regular classes such as `int`, `str`, `float`, and `bool`.
- Ignore complex annotations such as `List[int]`.
- Ignore missing annotations.
- Check the return value if a simple return annotation exists.

Raise `TypeError` with a helpful message when validation fails.

In [6]:
import inspect
from functools import wraps


def is_simple_type(annotation):
    return isinstance(annotation, type)


def enforce_simple_types(func):
    signature = inspect.signature(func)
    annotations = func.__annotations__

    @wraps(func)
    def wrapper(*args, **kwargs):
        bound = signature.bind(*args, **kwargs)
        bound.apply_defaults()

        for name, value in bound.arguments.items():
            annotation = annotations.get(name)
            if annotation is not None and is_simple_type(annotation):
                if not isinstance(value, annotation):
                    raise TypeError(
                        f"Argument {name!r} must be {annotation.__name__}, "
                        f"got {type(value).__name__}"
                    )

        result = func(*args, **kwargs)

        return_annotation = annotations.get('return')
        if return_annotation is not None and is_simple_type(return_annotation):
            if not isinstance(result, return_annotation):
                raise TypeError(
                    f"Return value must be {return_annotation.__name__}, "
                    f"got {type(result).__name__}"
                )

        return result

    return wrapper


@enforce_simple_types
def repeat(text: str, count: int) -> str:
    return text * count


# Verification
assert repeat('ha', 3) == 'hahaha'

try:
    repeat('ha', '3')
except TypeError as ex:
    error_message = str(ex)
else:
    error_message = 'No error raised'

error_message

"Argument 'count' must be int, got str"

## Problem 7: Ordered Named Tuple Factory with Defaults

### Task

Use ordered `**kwargs` to build a small named tuple factory.

Write `record_type(type_name, **fields_with_defaults)`.

Requirements:

1. Field order must match the order of keyword arguments.
2. The returned class should behave like a `namedtuple`.
3. Defaults should be applied when values are omitted.
4. Unknown fields should raise `TypeError`.

### Example

```python
User = record_type('User', id=None, name='anonymous', active=True)
User(id=1, name='Simeon')
```

In [7]:
from collections import namedtuple


def record_type(type_name, **fields_with_defaults):
    field_names = list(fields_with_defaults.keys())
    Base = namedtuple(type_name, field_names)

    class Record(Base):
        __slots__ = ()

        def __new__(cls, **kwargs):
            unknown = set(kwargs) - set(field_names)
            if unknown:
                unknown_fields = ', '.join(sorted(unknown))
                raise TypeError(f"Unknown field(s): {unknown_fields}")

            values = {
                name: kwargs.get(name, default)
                for name, default in fields_with_defaults.items()
            }

            return super(Record, cls).__new__(cls, **values)

    Record.__name__ = type_name
    return Record


# Verification
User = record_type('User', id=None, name='anonymous', active=True)
user = User(id=1, name='Simeon')

assert user.id == 1
assert user.name == 'Simeon'
assert user.active is True
assert User._fields == ('id', 'name', 'active')

try:
    User(role='admin')
except TypeError as ex:
    unknown_error = str(ex)
else:
    unknown_error = 'No error raised'

user, unknown_error

(User(id=1, name='Simeon', active=True), 'Unknown field(s): role')

## Problem 8: Async Batch Runner

### Task

Write an async batch runner using `async` and `await`.

Create:

1. An async function `fetch_value(name, delay, value)` that simulates an async operation.
2. An async function `run_batch(tasks)` that runs multiple tasks concurrently and returns an ordered dictionary-like result where keys appear in the original task order.

Each task should be a tuple:

```python
(name, delay, value)
```

The result should preserve task order, not completion order.

In [8]:
import asyncio


async def fetch_value(name, delay, value):
    await asyncio.sleep(delay)
    return name, value


async def run_batch(tasks):
    coroutines = [
        fetch_value(name, delay, value)
        for name, delay, value in tasks
    ]

    results = await asyncio.gather(*coroutines)
    return {name: value for name, value in results}


# Verification
# In a notebook, use: await run_batch(tasks)
# In a normal script, use: asyncio.run(run_batch(tasks))

tasks = [
    ('slow', 0.03, 100),
    ('fast', 0.01, 200),
    ('medium', 0.02, 300)
]

result = await run_batch(tasks)

assert list(result.keys()) == ['slow', 'fast', 'medium']
assert result == {'slow': 100, 'fast': 200, 'medium': 300}

result

{'slow': 100, 'fast': 200, 'medium': 300}

## Problem 9: f-string Expression Debug Report

### Task

Write `summarize_numbers(numbers)` using f-strings.

Return a multiline report containing:

- count
- minimum
- maximum
- average rounded to two decimal places
- total with comma separators

Requirements:

1. Use expressions directly inside f-strings.
2. Raise `ValueError` for an empty list.
3. Use numeric formatting mini-language features.

In [9]:
def summarize_numbers(numbers):
    if not numbers:
        raise ValueError('numbers must not be empty')

    return (
        f"count: {len(numbers)}\n"
        f"min: {min(numbers)}\n"
        f"max: {max(numbers)}\n"
        f"average: {sum(numbers) / len(numbers):.2f}\n"
        f"total: {sum(numbers):,}"
    )


# Verification
numbers = [1_000, 2_000, 3_000, 4_000]
report = summarize_numbers(numbers)

assert 'count: 4' in report
assert 'average: 2500.00' in report
assert 'total: 10,000' in report

print(report)

count: 4
min: 1000
max: 4000
average: 2500.00
total: 10,000


## Problem 10: Mini ETL System Combining Python 3.6 Features

### Task

Build a mini ETL pipeline using several Python 3.6 ideas together.

Write `transform_rows(rows, **operations)`.

Input:

```python
rows = [
    {'name': 'alice', 'score': 10},
    {'name': 'bob', 'score': 20}
]
```

`operations` is an ordered set of transformations where each key is the target field name and each value is a function receiving the full row.

Example:

```python
transform_rows(
    rows,
    display_name=lambda row: row['name'].title(),
    doubled_score=lambda row: row['score'] * 2
)
```

Requirements:

1. Preserve the original row keys first.
2. Add computed fields in the order given by `**operations`.
3. Do not mutate the original rows.
4. Use f-strings in at least one diagnostic message.
5. Include type annotations where reasonable.

In [10]:
from typing import Callable, Dict, Any, List


Row = Dict[str, Any]
Operation = Callable[[Row], Any]


def transform_rows(rows: List[Row], **operations: Operation) -> List[Row]:
    transformed = []

    for index, row in enumerate(rows):
        new_row = dict(row)

        for field_name, operation in operations.items():
            try:
                new_row[field_name] = operation(row)
            except Exception as ex:
                raise RuntimeError(
                    f"Failed to compute field {field_name!r} "
                    f"for row #{index}: {ex}"
                )

        transformed.append(new_row)

    return transformed


# Verification
rows = [
    {'name': 'alice', 'score': 10},
    {'name': 'bob', 'score': 20}
]

result = transform_rows(
    rows,
    display_name=lambda row: row['name'].title(),
    doubled_score=lambda row: row['score'] * 2,
    score_label=lambda row: f"score={row['score']}"
)

assert rows == [
    {'name': 'alice', 'score': 10},
    {'name': 'bob', 'score': 20}
]
assert list(result[0].keys()) == ['name', 'score', 'display_name', 'doubled_score', 'score_label']
assert result[0]['display_name'] == 'Alice'
assert result[1]['doubled_score'] == 40

result

[{'name': 'alice',
  'score': 10,
  'display_name': 'Alice',
  'doubled_score': 20,
  'score_label': 'score=10'},
 {'name': 'bob',
  'score': 20,
  'display_name': 'Bob',
  'doubled_score': 40,
  'score_label': 'score=20'}]

# Summary

This notebook practiced Python 3.6 features in advanced, realistic contexts:

- Ordered dictionaries for deterministic transformations
- Ordered `**kwargs` for APIs and factories
- Numeric literal underscores for readability
- f-strings for formatting and diagnostics
- Type annotations for documentation and tooling
- Runtime inspection of annotations
- Async execution with order-preserving results

A strong Python 3.6 codebase uses these features to make code more readable, deterministic, and easier to maintain.